# FedX-PALM: Federated Learning Simulation (4 Clients + DP + XAI)

**Master's Thesis - Rachma Dianty (201012420007)** · Telkom University, 2026

---

## Tujuan
Notebook ini menjalankan **simulasi Federated Learning** sesuai inti tesis FedX-Palm:

1. **4 client node** dengan split Non-IID Dirichlet (α = 0.1, 0.3, 0.5, 0.7)
2. **FedAvg** weighted by client sample count
3. **Differential Privacy** (DP-FedAvg — clipping + Gaussian noise pada Δw)
4. **4 skenario ε** (baseline / weak 8.0 / moderate 4.0 / strong 1.0)
5. **Grad-CAM++ + Average Drop + FRR** pada model final
6. Plot **privacy-utility trade-off**

## Prerequisite
Notebook ini **butuh `best.pt`** dari notebook centralized sebelumnya yang sudah tersimpan di Drive (`/content/drive/MyDrive/fedx-palm/centralized_baseline/best.pt`). Kalau belum ada, jalankan dulu `fedx_palm_centralized_training.ipynb`.

## Catatan implementasi DP
Tesis Bab 3.4 menyebut Opacus DP-SGD untuk per-sample gradient noise. Namun **Opacus tidak kompatibel langsung dengan Ultralytics YOLOv11** (BatchNorm + mosaic + closed trainer). Implementasi di sini pakai **DP-FedAvg** (McMahan et al. 2018) — clipping + Gaussian noise pada *delta weight per klien*. Secara matematis tetap memberikan (ε, δ)-DP guarantee di level user/round, dan ini approach yang umum dipakai di FL+detection literatur.

## 1. Setup environment

In [1]:
!pip install -q ultralytics==8.4.51 roboflow grad-cam pyyaml opencv-python-headless seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 44.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 123.1 MB/s eta 0:00:00


In [2]:
import torch, ultralytics, roboflow
from ultralytics import YOLO
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Ultralytics : {ultralytics.__version__}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch     : 2.10.0+cu128
CUDA        : True
GPU         : Tesla T4
VRAM        : 15.6 GB
Ultralytics : 8.4.51


## 2. Mount Drive & load baseline weight

In [3]:
import os, shutil
from pathlib import Path

# === Auto-detect Colab vs Local (Windows/Linux) ===
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    drive.mount("/content/drive")
    BASELINE_W0 = "/content/drive/MyDrive/fedx-palm/centralized_baseline/best.pt"
else:
    # LOCAL. Letakkan best.pt di path ini, atau set env var BASELINE_W0.
    DEFAULT_LOCAL = r"D:\rachma\fedx-palm-weights\best.pt"
    BASELINE_W0 = os.environ.get("BASELINE_W0", DEFAULT_LOCAL)
    # /content jadi <drive saat ini>\content di Windows
    for sub in ["/content", "/content/data", "/content/sim_runs"]:
        os.makedirs(sub, exist_ok=True)

assert os.path.exists(BASELINE_W0), (
    f"best.pt tidak ditemukan: {BASELINE_W0}\n"
    f"  - Colab: jalankan notebook centralized dulu.\n"
    f"  - Local: letakkan best.pt di path tsb atau set env var BASELINE_W0."
)

W0_LOCAL = "/content/w0.pt"
shutil.copy2(BASELINE_W0, W0_LOCAL)
print(f"Env          : {'Colab' if IS_COLAB else 'Local (Windows/Linux)'}")
print(f"Baseline src : {BASELINE_W0}")
print(f"Initial w0   : {W0_LOCAL} ({os.path.getsize(W0_LOCAL)//1024} KB)")


Mounted at /content/drive
Initial weight w0: /content/w0.pt (5345 KB)


## 3. Download dataset Roboflow

In [4]:
from roboflow import Roboflow
rf = Roboflow(api_key='Ej0bSMpeSri3ky0IYkOU')
project = rf.workspace('dydy-worker').project('palm-fruit-ripeness-detection-f6sac-ccb2z')
version = project.version(2)
dataset = version.download('yolov11', location='/content/data/palm_v2')
RAW_DIR = dataset.location
print(f'Dataset: {RAW_DIR}')

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/palm_v2 in yolov11:: 100%|██████████| 21640/21640 [00:04<00:00, 4480.30it/s]


Dataset: /content/data/palm_v2


In [5]:
import yaml
with open(f'{RAW_DIR}/data.yaml') as f:
    raw_meta = yaml.safe_load(f)
CLASS_NAMES = raw_meta['names']
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]
NC = len(CLASS_NAMES)
print(f'Classes ({NC}): {CLASS_NAMES}')

Classes (6): ['Abnormal', 'Empty Bunch', 'Overripe', 'Ripe', 'Underripe', 'Unripe']


## 3b. Re-split berbasis bunch_id (R4 — fix Pembimbing II)

Roboflow membagi train/valid/test secara acak **per-frame**, padahal beberapa frame
berasal dari **tandan yang sama** (sumber: video). Akibatnya satu tandan dapat
muncul di train sekaligus valid sehingga model "menghafal" tandan, bukan belajar
generalisasi. Audit di sel 4b sebelumnya menunjukkan SOFT leakage 100%.

**Strategi anti-leakage:**

- **Group key**: `bunch_id` diekstrak dari nama file
  (`framesawit27-7-_png.rf.<hash>.jpg` → `framesawit27`).
- **Stratifikasi**: per kelas dominan dari setiap tandan, agar kelas langka
  (mis. *Empty Bunch*) tetap terwakili di valid/test.
- **Rasio target**: 80/10/10 (mempertahankan rasio Roboflow asli).
- **Garansi**: satu `bunch_id` hanya muncul di **satu** split.
- **Seed**: 42 (deterministik & reproducible).

Setelah re-split, variabel `RAW_DIR` diarahkan ke direktori bersih sehingga
seluruh pipeline hilir (Dirichlet client split, training, evaluasi global)
otomatis menggunakan data tanpa leakage.


In [ ]:
# === RE-SPLIT BERBASIS bunch_id (R4 fix) ===
import os, re, shutil, random
from pathlib import Path
from collections import defaultdict, Counter

ORIG_DIR     = Path(RAW_DIR)                          # data asli Roboflow (leaky)
RESPLIT_DIR  = Path('/content/data/palm_v2_resplit')  # target bersih
RATIO        = {'train': 0.80, 'valid': 0.10, 'test': 0.10}
RESPLIT_SEED = 42

def _strip_rf(p): return re.split(r'\.rf\.', os.path.basename(p))[0]
def _bunch_id(p):
    b = _strip_rf(p); m = re.match(r'(frame[a-z]*\d+)', b, re.I)
    return m.group(1) if m else b
def _primary_cls(lbl_path):
    if not lbl_path.exists(): return -1
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts: return int(parts[0])
    return -1

# 1. Pool semua image dari 3 split asli
pairs = []
for split in ['train', 'valid', 'test']:
    img_dir = ORIG_DIR / split / 'images'
    lbl_dir = ORIG_DIR / split / 'labels'
    if not img_dir.exists(): continue
    for img in img_dir.iterdir():
        if img.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
        pairs.append((img, lbl_dir / (img.stem + '.txt')))
print(f'Populasi gabungan : {len(pairs)} citra')

# 2. Group by bunch_id
bunches = defaultdict(list)
for img, lbl in pairs:
    bunches[_bunch_id(img)].append((img, lbl))
print(f'Total tandan unik : {len(bunches)}')

# 3. Kelas dominan per tandan (untuk stratifikasi)
bunch_cls = {bid: Counter(_primary_cls(l) for _, l in items).most_common(1)[0][0]
             for bid, items in bunches.items()}
dist_pretty = dict(Counter(
    (CLASS_NAMES[c] if 0 <= c < NC else 'unlabeled') for c in bunch_cls.values()))
print(f'Distribusi tandan per kelas dominan: {dist_pretty}')

# 4. Stratified group split: per kelas, shuffle deterministik, 80/10/10
rng = random.Random(RESPLIT_SEED)
by_cls = defaultdict(list)
for bid, c in bunch_cls.items():
    by_cls[c].append(bid)

split_assign = {}
for c, bids in sorted(by_cls.items()):
    rng.shuffle(bids)
    n = len(bids)
    n_tr = int(round(n * RATIO['train']))
    n_va = int(round(n * RATIO['valid']))
    if n >= 3:                       # garansi >=1 tandan di tiap split
        n_tr = max(1, min(n_tr, n - 2))
        n_va = max(1, n_va)
    elif n == 2:                     # 1 train, 1 valid
        n_tr, n_va = 1, 1
    else:                            # n == 1: train-only (warn)
        n_tr, n_va = 1, 0
        cname = CLASS_NAMES[c] if 0 <= c < NC else 'unlabeled'
        print(f'  ! kelas "{cname}" hanya 1 tandan -> train-only')
    for bid in bids[:n_tr]:                split_assign[bid] = 'train'
    for bid in bids[n_tr:n_tr + n_va]:     split_assign[bid] = 'valid'
    for bid in bids[n_tr + n_va:]:         split_assign[bid] = 'test'

# 5. Materialize direktori baru
if RESPLIT_DIR.exists(): shutil.rmtree(RESPLIT_DIR)
for s in ['train', 'valid', 'test']:
    (RESPLIT_DIR / s / 'images').mkdir(parents=True, exist_ok=True)
    (RESPLIT_DIR / s / 'labels').mkdir(parents=True, exist_ok=True)

cnt = Counter(); cls_dist = defaultdict(Counter)
for bid, items in bunches.items():
    s = split_assign[bid]
    for img, lbl in items:
        shutil.copy2(img, RESPLIT_DIR / s / 'images' / img.name)
        if lbl.exists():
            shutil.copy2(lbl, RESPLIT_DIR / s / 'labels' / lbl.name)
        cnt[s] += 1
        cls_dist[s][_primary_cls(lbl)] += 1

# 6. Tulis data.yaml baru
with open(RESPLIT_DIR / 'data.yaml', 'w') as f:
    yaml.safe_dump({
        'path':  str(RESPLIT_DIR.absolute()),
        'train': 'train/images',
        'val':   'valid/images',
        'test':  'test/images',
        'nc':    NC,
        'names': CLASS_NAMES,
    }, f, sort_keys=False)

print('\nHasil re-split (per tandan, stratified):')
for s in ['train', 'valid', 'test']:
    pretty = {(CLASS_NAMES[c] if 0 <= c < NC else 'unlabeled'): cls_dist[s][c]
              for c in sorted(cls_dist[s])}
    n_b = sum(1 for _, sp in split_assign.items() if sp == s)
    print(f'  {s:6}: {cnt[s]:5d} citra | {n_b:4d} tandan | {pretty}')

# 7. Arahkan RAW_DIR ke split bersih
RAW_DIR = str(RESPLIT_DIR)
print(f'\nRAW_DIR -> {RAW_DIR}')
print('Pipeline hilir (Dirichlet split, training, evaluasi) sekarang pakai data bersih.')


## 4. Split Non-IID Dirichlet

4 client dengan α berbeda merepresentasikan heterogenitas perkebunan (Tesis Tabel 3.3). Validation & test set **shared** — semua klien dievaluasi di set yang sama supaya perbandingan fair.

In [6]:
import numpy as np, random, shutil
from pathlib import Path
from collections import defaultdict, Counter

SEED = 42
np.random.seed(SEED); random.seed(SEED)

ALPHA_PER_CLIENT = {1: 0.1, 2: 0.3, 3: 0.5, 4: 0.7}
CLIENT_DATA_ROOT = Path('/content/data/clients')
if CLIENT_DATA_ROOT.exists():
    shutil.rmtree(CLIENT_DATA_ROOT)

raw_train_img = Path(RAW_DIR) / 'train' / 'images'
raw_train_lbl = Path(RAW_DIR) / 'train' / 'labels'
raw_val_img   = Path(RAW_DIR) / 'valid' / 'images'
raw_val_lbl   = Path(RAW_DIR) / 'valid' / 'labels'

# Group training images by primary class (first annotation)
def primary_class(lbl_path):
    if not lbl_path.exists(): return 0
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts: return int(parts[0])
    return 0

by_class = defaultdict(list)
for img in raw_train_img.iterdir():
    if img.suffix.lower() not in {'.jpg', '.jpeg', '.png'}: continue
    cls = primary_class(raw_train_lbl / (img.stem + '.txt'))
    by_class[cls].append(img)
for c in by_class: random.shuffle(by_class[c])
print('Total images per class:', {CLASS_NAMES[c]: len(by_class[c]) for c in sorted(by_class)})

Total images per class: {'Abnormal': 1644, 'Empty Bunch': 645, 'Overripe': 1706, 'Ripe': 1932, 'Underripe': 1671, 'Unripe': 2384}


In [7]:
# Sample Dirichlet proportions per client, then allocate images per class
client_proportions = {}
for k, alpha in ALPHA_PER_CLIENT.items():
    client_proportions[k] = np.random.dirichlet([alpha] * NC)
    print(f'Client {k} (α={alpha}): {np.round(client_proportions[k], 3)}')

client_files = {k: [] for k in ALPHA_PER_CLIENT}
for cls in sorted(by_class):
    imgs = by_class[cls]
    n = len(imgs)
    props = np.array([client_proportions[k][cls] for k in ALPHA_PER_CLIENT])
    props = props / props.sum()
    allocated = 0
    for i, k in enumerate(ALPHA_PER_CLIENT):
        if i == len(ALPHA_PER_CLIENT) - 1:
            n_alloc = n - allocated
        else:
            n_alloc = int(round(props[i] * n))
            n_alloc = min(n_alloc, n - allocated)
        client_files[k].extend(imgs[allocated:allocated + n_alloc])
        allocated += n_alloc

for k in ALPHA_PER_CLIENT:
    cnt = Counter()
    for img in client_files[k]:
        cnt[primary_class(raw_train_lbl / (img.stem + '.txt'))] += 1
    pretty = {CLASS_NAMES[c]: cnt[c] for c in sorted(cnt)}
    print(f'Client {k} (α={ALPHA_PER_CLIENT[k]}): {len(client_files[k])} img | {pretty}')

Client 1 (α=0.1): [      0.001       0.877           0           0       0.122           0]
Client 2 (α=0.3): [      0.787       0.004       0.023       0.075        0.02        0.09]
Client 3 (α=0.5): [      0.009       0.087       0.001       0.762       0.021        0.12]
Client 4 (α=0.7): [      0.011       0.002       0.032       0.041       0.107       0.808]
Client 1 (α=0.1): 1342 img | {'Abnormal': 2, 'Empty Bunch': 583, 'Underripe': 757}
Client 2 (α=0.3): 2815 img | {'Abnormal': 1601, 'Empty Bunch': 3, 'Overripe': 710, 'Ripe': 165, 'Underripe': 126, 'Unripe': 210}
Client 3 (α=0.5): 2191 img | {'Abnormal': 18, 'Empty Bunch': 58, 'Overripe': 29, 'Ripe': 1676, 'Underripe': 129, 'Unripe': 281}
Client 4 (α=0.7): 3634 img | {'Abnormal': 23, 'Empty Bunch': 1, 'Overripe': 967, 'Ripe': 91, 'Underripe': 659, 'Unripe': 1893}


In [8]:
# Materialize each client folder + data.yaml
for k in ALPHA_PER_CLIENT:
    cdir = CLIENT_DATA_ROOT / f'client_{k}'
    (cdir / 'images' / 'train').mkdir(parents=True, exist_ok=True)
    (cdir / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
    (cdir / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (cdir / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
    for img in client_files[k]:
        shutil.copy2(img, cdir / 'images' / 'train' / img.name)
        lbl = raw_train_lbl / (img.stem + '.txt')
        if lbl.exists(): shutil.copy2(lbl, cdir / 'labels' / 'train' / lbl.name)
    # shared val set (kecil saja: 50 random img dari raw val)
    val_imgs = list(raw_val_img.iterdir())[:50]
    for img in val_imgs:
        shutil.copy2(img, cdir / 'images' / 'val' / img.name)
        lbl = raw_val_lbl / (img.stem + '.txt')
        if lbl.exists(): shutil.copy2(lbl, cdir / 'labels' / 'val' / lbl.name)
    with open(cdir / 'data.yaml', 'w') as f:
        yaml.safe_dump({
            'path': str(cdir.absolute()),
            'train': 'images/train',
            'val': 'images/val',
            'nc': NC,
            'names': CLASS_NAMES,
        }, f, sort_keys=False)
    print(f'  Client {k} ready: {cdir}')

# Shared global validation yaml (untuk evaluasi global per ronde, pakai full val set)
GLOBAL_VAL_YAML = Path('/content/data/global_val.yaml')
with open(GLOBAL_VAL_YAML, 'w') as f:
    yaml.safe_dump({
        'path': str(Path(RAW_DIR).absolute()),
        'train': 'valid/images',  # YOLO requires both keys; reuse val
        'val':   'valid/images',
        'nc': NC,
        'names': CLASS_NAMES,
    }, f, sort_keys=False)
print(f'\nGlobal val yaml: {GLOBAL_VAL_YAML}')

  Client 1 ready: /content/data/clients/client_1
  Client 2 ready: /content/data/clients/client_2
  Client 3 ready: /content/data/clients/client_3
  Client 4 ready: /content/data/clients/client_4

Global val yaml: /content/data/global_val.yaml


## 4b. Audit Data Leakage — Verifikasi (R4)

Verifikasi bahwa sel 3b berhasil menghapus leakage: tidak ada `bunch_id` yang
sama bocor antar split (train ↔ valid, train ↔ test, valid ↔ test). Audit
diarahkan ke `RAW_DIR` aktif — jadi kalau sel 3b sudah dijalankan, audit ini
mengevaluasi split bersih; kalau sel 3b di-skip, audit mengevaluasi split
Roboflow asli yang leaky.


In [ ]:
# === AUDIT DATA LEAKAGE (R4) ===
import os, re, glob

# Prefer RAW_DIR (mungkin sudah diarahkan ke resplit di sel 3b)
DATASET_DIR = None
try:
    if os.path.isdir(os.path.join(RAW_DIR, 'train', 'images')):
        DATASET_DIR = RAW_DIR
except NameError: pass
if DATASET_DIR is None:
    for c in ['/content/data/palm_v2', '/content/datasets/palm_v2']:
        if os.path.isdir(os.path.join(c, 'train', 'images')):
            DATASET_DIR = c; break
if DATASET_DIR is None:
    hits = glob.glob('/content/**/train/images', recursive=True)
    if hits: DATASET_DIR = os.path.dirname(os.path.dirname(hits[0]))
assert DATASET_DIR, 'Dataset dir tidak ketemu - set manual DATASET_DIR'
print('DATASET_DIR:', DATASET_DIR)

def strip_rf(p): return re.split(r'\.rf\.', os.path.basename(p))[0]
def bunch_id(p):
    b = strip_rf(p); m = re.match(r'(frame[a-z]*\d+)', b, re.I)
    return m.group(1) if m else b
def collect(split):
    ps = glob.glob(f'{DATASET_DIR}/{split}/images/*.jpg')
    return ps, {strip_rf(x) for x in ps}, {bunch_id(x) for x in ps}

tr_p, tr_o, tr_b = collect('train')
va_p, va_o, va_b = collect('valid')
te_p, te_o, te_b = collect('test')
for nm, (p, o, b) in [('train',(tr_p,tr_o,tr_b)),
                       ('valid',(va_p,va_o,va_b)),
                       ('test', (te_p,te_o,te_b))]:
    print(f'{nm:6}: {len(p):5d} citra | {len(o):5d} original unik | {len(b):4d} tandan unik')
print()

pairs_check = [('train', 'valid', tr_o, va_o, tr_b, va_b),
               ('train', 'test',  tr_o, te_o, tr_b, te_b),
               ('valid', 'test',  va_o, te_o, va_b, te_b)]
total_hard = total_soft = 0
for a, b, ao, bo, ab, bb in pairs_check:
    hard, soft = ao & bo, ab & bb
    total_hard += len(hard); total_soft += len(soft)
    print(f'  [{a:5} ∩ {b:5}] HARD: {len(hard):3d} citra | SOFT: {len(soft):3d} tandan')
    if soft: print(f'      contoh tandan bocor: {list(soft)[:3]}')

print()
if total_hard == 0 and total_soft == 0:
    print('>>> BERSIH: tidak ada leakage di semua pasangan split.')
elif total_hard > 0:
    print('>>> LEAKAGE BERAT: citra identik di antara split. Jalankan sel 3b utk re-split.')
else:
    print('>>> SOFT leakage masih ada. Jalankan sel 3b untuk re-split per bunch_id.')


## 5. Federated Learning engine

Fungsi-fungsi inti: train satu klien, apply DP pada delta weight, FedAvg aggregation, evaluasi global.

In [ ]:
import copy, math, json, time
import torch
from ultralytics import YOLO   # defensive: re-import bila kernel restart

# =============================================================================
# DP-FedAvg (client-level DP) — McMahan et al. (2018)
# Tiap klien melatih SGD normal, lalu delta bobot (w_local - w_global) di-clip ke
# L2 norm C dan ditambah Gaussian noise N(0, (sigma*C)^2) per elemen SEBELUM
# dikirim ke server. Ini BUKAN Opacus per-sample DP-SGD (Opacus tidak kompatibel
# dengan BatchNorm + closed trainer Ultralytics YOLOv11).
# =============================================================================
# REVISI (Evaluasi Pembimbing II, R1-R3):
#   * sigma lama (0.005/0.01/0.02): noise/elem = sigma*C = 0.05..0.20, JAUH lebih
#     besar dari sinyal/elem (~C/sqrt(d) = 10/sqrt(2.59e6) ~ 0.006). Akibatnya
#     model collapse total (mAP=0) di SEMUA sigma -> artefak kalibrasi, bukan
#     trade-off privasi sejati.
#   * Sweep ulang sigma jauh lebih kecil agar transisi gradual terlihat.
#   * epsilon DIHITUNG dari accountant RDP (numpy, mandiri), bukan dilabel
#     manual dan TIDAK memanggil Opacus.
# =============================================================================

DELTA_DP = 1e-5   # target delta untuk (epsilon, delta)-DP

def epsilon_from_sigma(sigma, T, delta=DELTA_DP):
    """(epsilon, delta)-DP untuk komposisi T-ronde Gaussian mechanism (q=1).
    Noise multiplier z = sigma. Implementasi mandiri (numpy), independen dari
    Opacus. Primer: RDP accountant (Mironov 2017); fallback: zCDP analitik
    (Bun & Steinke 2016). Definisi penuh: lihat sel 6b 'Derivasi epsilon'."""
    if sigma is None or sigma <= 0:
        return float('inf')
    try:
        import numpy as _np
        a = _np.concatenate([_np.arange(1.01, 100, 0.01), _np.arange(100, 1e6, 100)])
        eps = T * a / (2.0 * sigma**2) + _np.log(1.0/delta) / (a - 1.0)
        return float(eps.min())
    except Exception:
        rho = T / (2.0 * sigma**2)                       # zCDP komposisi T Gaussian
        return rho + 2.0*math.sqrt(rho*math.log(1.0/delta))

# Sweep ulang. epsilon diisi otomatis di run_scenario memakai ROUNDS aktual.
PRIVACY_SCENARIOS = {
    'baseline':   {'noise_multiplier': 0.0,    'clip_norm': 0.0 },
    'sigma_1em4': {'noise_multiplier': 0.0001, 'clip_norm': 10.0},
    'sigma_3em4': {'noise_multiplier': 0.0003, 'clip_norm': 10.0},
    'sigma_5em4': {'noise_multiplier': 0.0005, 'clip_norm': 10.0},
    'sigma_1em3': {'noise_multiplier': 0.001,  'clip_norm': 10.0},
    'sigma_3em3': {'noise_multiplier': 0.003,  'clip_norm': 10.0},
    # --- ABLATION opsional (R2). Aktifkan utk uji penyebab collapse: ---
    # 'clip_only':  {'noise_multiplier': 0.0,    'clip_norm': 10.0},  # clip tanpa noise -> harus ~baseline
    # 'sigma_big':  {'noise_multiplier': 0.5,    'clip_norm': 10.0},  # noise standar literatur -> cek NaN/collapse
}

def yolo_state(model):
    return {k: v.detach().clone().cpu() for k, v in model.model.state_dict().items()}

def load_state(model, state):
    tgt = model.model.state_dict()
    new = {}
    for k, v in state.items():
        if k in tgt:
            new[k] = v.to(tgt[k].device).to(tgt[k].dtype)
    tgt.update(new)
    model.model.load_state_dict(tgt, strict=False)

def apply_dp_to_delta(delta, clip_norm, noise_mult):
    """Client-level DP pada delta weight. Return (privatized_delta, diag_info)."""
    flat = torch.cat([v.float().flatten() for v in delta.values()])
    pre_norm = flat.norm(p=2).item()
    d = flat.numel()
    if clip_norm <= 0 or noise_mult <= 0:
        info = {'pre_norm': pre_norm, 'd': d, 'clipped': False,
                'signal_per_elem': pre_norm/(d**0.5 + 1e-12), 'noise_per_elem': 0.0}
        return delta, info
    scale = min(1.0, clip_norm / (pre_norm + 1e-12))
    eff_norm = pre_norm * scale
    noise_per_elem = noise_mult * clip_norm
    out = {}
    for k, v in delta.items():
        v_c = v.float() * scale
        noise = torch.randn_like(v_c) * noise_per_elem
        out[k] = (v_c + noise).to(v.dtype)
    info = {'pre_norm': pre_norm, 'd': d, 'clipped': scale < 1.0,
            'signal_per_elem': eff_norm/(d**0.5 + 1e-12), 'noise_per_elem': noise_per_elem}
    return out, info

def train_client(w0_path, data_yaml, local_epochs, run_name, output_dir, img=640, batch=16, lr=0.01):
    model = YOLO(w0_path)
    model.train(
        data=str(data_yaml), epochs=local_epochs, imgsz=img, batch=batch,
        optimizer='SGD', lr0=lr, device=0, workers=2, cache=False,
        project=str(output_dir), name=run_name, exist_ok=True,
        verbose=False, plots=False, save=True, seed=SEED,
    )
    return yolo_state(model)

def eval_global(weights_path, val_yaml, img=640, batch=16):
    m = YOLO(weights_path)
    r = m.val(data=str(val_yaml), imgsz=img, batch=batch, device=0,
              workers=2, verbose=False, plots=False, save_json=False)
    return {
        'mAP_50':    float(r.box.map50),
        'mAP_50_95': float(r.box.map),
        'precision': float(r.box.mp),
        'recall':    float(r.box.mr),
    }

print('FL engine (DP-FedAvg client-level) ready.')


## 6. Konfigurasi simulasi

In [ ]:
ROUNDS         = 5      # tesis terkoreksi: 5 ronde komunikasi
LOCAL_EPOCHS   = 2       # tesis terkoreksi: 2 epoch lokal/ronde
BATCH_SIZE     = 16
IMG_SIZE       = 640
LR0            = 0.01
# Sweep ulang (Eval. Pembimbing II R2/R3). Mulai dari subset 3 skenario kalau
# waktu Colab mepet, lalu lengkapi. Tiap skenario ~ 5 ronde x 4 klien x 2 epoch.
SCENARIOS      = ['baseline', 'sigma_1em4', 'sigma_3em4', 'sigma_5em4', 'sigma_1em3', 'sigma_3em3']
SIM_OUTPUT     = Path('/content/sim_runs')
SIM_OUTPUT.mkdir(exist_ok=True)

# Pre-count client sample sizes untuk weighted FedAvg
CLIENT_SIZES = {k: len(client_files[k]) for k in ALPHA_PER_CLIENT}
TOTAL_N = sum(CLIENT_SIZES.values())
weights = {k: round(n/TOTAL_N, 3) for k, n in CLIENT_SIZES.items()}
print(f'Total samples across clients: {TOTAL_N}')
print(f'FedAvg weights: {weights}')

# Pratinjau epsilon terhitung untuk tiap skenario (T=ROUNDS, delta=DELTA_DP)
print("\\nEpsilon terhitung (accountant RDP/zCDP):")
for _s in SCENARIOS:
    _sig = PRIVACY_SCENARIOS[_s]['noise_multiplier']
    print(f"  {_s:12s} sigma={_sig:<8} -> eps={epsilon_from_sigma(_sig, ROUNDS):.1f}")

## 6b. Derivasi Privacy Budget ε dari Accountant (R1 — Pembimbing II)

ε **tidak** dilabel manual; dihitung dari privacy accountant berdasarkan
(σ, q, T, δ). Mekanisme: **DP-FedAvg level-klien** (Gaussian, noise multiplier
z = σ), partisipasi penuh q = 1.0, komposisi T = `ROUNDS` ronde, δ = `DELTA_DP`.

- **RDP (Mironov, 2017):** `ε = min_{α>1} [ T·α/(2σ²) + ln(1/δ)/(α−1) ]`
  — diimplementasikan **mandiri** (numpy), independen dari trainer Opacus
  yang tidak digunakan pada notebook ini (Opacus tidak kompatibel dengan
  BatchNorm YOLOv11; lihat catatan implementasi di awal notebook).
- **Cross-check zCDP** analitik (Bun & Steinke, 2016) sebagai validasi kedua.

Sel ini mencetak: (a) ε terhitung tiap skenario, (b) σ yang dibutuhkan untuk
ε target, dan menyimpan `privacy_budget_derivation.csv` untuk lampiran tesis.


In [ ]:
# === DERIVASI PRIVACY BUDGET epsilon (R1) ===
# Accountant mandiri (numpy) — independen dari Opacus. Trainer Opacus tidak
# dipakai pada notebook ini karena auto-fix BatchNorm-nya berisiko mengubah
# arsitektur YOLOv11 secara diam-diam (lihat catatan Pembimbing II, R2).
import numpy as np, json, math
from pathlib import Path

Q_SAMPLING = 1.0   # partisipasi penuh 4 klien tiap ronde -> tanpa amplifikasi subsampling

def epsilon_rdp_exact(sigma, T, delta=DELTA_DP):
    """(eps,delta)-DP utk Gaussian mechanism non-subsampled (q=1), komposisi T ronde.
    RDP @ order a: eps_RDP(a)=T*a/(2 sigma^2). Konversi: +ln(1/delta)/(a-1), min over a.
    Ref: Mironov (2017), 'Renyi Differential Privacy'."""
    if sigma is None or sigma <= 0:
        return float('inf')
    a = np.concatenate([np.arange(1.01, 100, 0.01), np.arange(100, 1e6, 100)])
    eps = T * a / (2.0 * sigma**2) + np.log(1.0/delta) / (a - 1.0)
    return float(eps.min())

def epsilon_zcdp(sigma, T, delta=DELTA_DP):
    """Cross-check zCDP analitik. Ref: Bun & Steinke (2016)."""
    if sigma is None or sigma <= 0: return float('inf')
    rho = T / (2.0 * sigma**2)
    return rho + 2.0*math.sqrt(rho*math.log(1.0/delta))

def sigma_for_target_eps(target_eps, T, delta=DELTA_DP):
    """Bisection: cari sigma sehingga RDP-accountant memberi eps == target."""
    lo, hi = 1e-4, 1e4
    for _ in range(200):
        mid = math.sqrt(lo*hi)
        if epsilon_rdp_exact(mid, T, delta) > target_eps: lo = mid
        else: hi = mid
    return math.sqrt(lo*hi)

print(f'Parameter: T(ronde)={ROUNDS}  q(sampling rate)={Q_SAMPLING}  delta={DELTA_DP}\n')

# (a) epsilon per skenario sweep
rows = []
print(f'{"skenario":12s} {"sigma":>8s} {"eps_RDP":>12s} {"eps_zCDP":>12s}  rezim')
for s, cfg in PRIVACY_SCENARIOS.items():
    sig = cfg['noise_multiplier']
    e_rdp = epsilon_rdp_exact(sig, ROUNDS)
    e_zcdp = epsilon_zcdp(sig, ROUNDS)
    rezim = 'no-privacy (eps>>10)' if e_rdp > 10 else ('lemah' if e_rdp > 4 else 'bermakna')
    rdp_str = 'inf' if math.isinf(e_rdp) else f'{e_rdp:.3e}'
    zc_str  = 'inf' if math.isinf(e_zcdp) else f'{e_zcdp:.3e}'
    print(f'{s:12s} {sig:8.4f} {rdp_str:>12s} {zc_str:>12s}  {rezim}')
    rows.append({'scenario': s, 'sigma': sig, 'q': Q_SAMPLING, 'T': ROUNDS,
                 'delta': DELTA_DP, 'eps_rdp': e_rdp, 'eps_zcdp': e_zcdp,
                 'regime': rezim})

# (b) sigma yang dibutuhkan untuk privasi bermakna
print('\nsigma yang DIBUTUHKAN untuk eps target (rezim privasi bermakna):')
for tgt in [8.0, 4.0, 1.0]:
    print(f'  target eps={tgt:<5} -> butuh sigma ~ {sigma_for_target_eps(tgt, ROUNDS):.3f}')

# (c) simpan untuk lampiran tesis
out = Path(SIM_OUTPUT) / 'privacy_budget_derivation.csv'
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w') as f:
    f.write('scenario,sigma,q,T,delta,eps_rdp,eps_zcdp,regime\n')
    for r in rows:
        f.write(f"{r['scenario']},{r['sigma']},{r['q']},{r['T']},{r['delta']},"
                f"{r['eps_rdp']:.6e},{r['eps_zcdp']:.6e},{r['regime']}\n")
print(f'\nDerivasi tersimpan: {out}')
print('\nKESIMPULAN R1: sigma yang dipakai memberi eps >> 10^3 (tanpa privasi bermakna).')
print('Privasi bermakna (eps~1-8) butuh sigma~1.5-11, namun di rentang itu model collapse (lihat R2/R3).')


## 7. Loop simulasi per skenario ε

Untuk tiap skenario: jalankan ROUNDS ronde × 4 client. Setelah tiap ronde, evaluasi global mAP di shared val set, lalu broadcast model agregat ke ronde berikut.

**Estimasi waktu**: ~5 menit/client/ronde × 4 client × 20 ronde × 4 skenario ≈ 26 jam. **Terlalu lama untuk Colab Pro biasa.**

Disarankan jalankan **1 skenario per session Colab** (ubah `SCENARIOS = ['baseline']`, lalu next session `['weak']`, dst). Atau turunkan `ROUNDS` ke 5–10 untuk smoke test dulu.

In [ ]:
def run_scenario(scenario_name):
    privacy = PRIVACY_SCENARIOS[scenario_name]
    # R1: hitung epsilon dari accountant pakai ROUNDS aktual (bukan label manual)
    privacy['epsilon'] = epsilon_from_sigma(privacy['noise_multiplier'], ROUNDS)
    print(f"\n{'='*64}")
    print(f"SCENARIO: {scenario_name} | sigma={privacy['noise_multiplier']} "
          f"C={privacy['clip_norm']} -> eps={privacy['epsilon']:.1f} (delta={DELTA_DP})")
    print('='*64)

    scen_dir = SIM_OUTPUT / scenario_name
    scen_dir.mkdir(parents=True, exist_ok=True)
    global_w_path = str(scen_dir / 'global.pt')

    shutil.copy2(W0_LOCAL, global_w_path)
    init_model = YOLO(global_w_path)
    global_state = yolo_state(init_model)
    del init_model; torch.cuda.empty_cache()

    first_client = list(ALPHA_PER_CLIENT)[0]
    history = []
    for r in range(1, ROUNDS + 1):
        t0 = time.time()
        client_deltas = []
        for k in ALPHA_PER_CLIENT:
            local_state = train_client(
                global_w_path,
                CLIENT_DATA_ROOT / f'client_{k}' / 'data.yaml',
                LOCAL_EPOCHS,
                run_name=f'r{r:02d}_c{k}',
                output_dir=scen_dir / 'runs',
                img=IMG_SIZE, batch=BATCH_SIZE, lr=LR0,
            )
            delta = {key: (local_state[key] - global_state[key]) for key in global_state}
            delta, dp_info = apply_dp_to_delta(delta, privacy['clip_norm'], privacy['noise_multiplier'])
            # R2 diagnostik: rasio noise/sinyal (bukti penyebab collapse) pd ronde-1 klien-1
            if r == 1 and k == first_client and privacy['noise_multiplier'] > 0:
                ratio = dp_info['noise_per_elem'] / (dp_info['signal_per_elem'] + 1e-12)
                print(f"    [DP diag] pre_clip_norm={dp_info['pre_norm']:.3f} clipped={dp_info['clipped']} "
                      f"| signal/elem={dp_info['signal_per_elem']:.2e} noise/elem={dp_info['noise_per_elem']:.2e} "
                      f"| noise/signal={ratio:.1f}x")
            client_deltas.append(delta)
            torch.cuda.empty_cache()

        # FedAvg weighted by client_sizes
        new_state = {k: v.clone() for k, v in global_state.items()}
        for key in new_state:
            agg = torch.zeros_like(new_state[key], dtype=torch.float32)
            for (k_id, n_k), delta in zip(CLIENT_SIZES.items(), client_deltas):
                agg += (n_k / TOTAL_N) * delta[key].float()
            new_state[key] = (new_state[key].float() + agg).to(new_state[key].dtype)
        global_state = new_state

        agg_model = YOLO(W0_LOCAL)
        load_state(agg_model, global_state)
        agg_model.save(global_w_path)
        del agg_model; torch.cuda.empty_cache()

        m = eval_global(global_w_path, GLOBAL_VAL_YAML, img=IMG_SIZE, batch=BATCH_SIZE)
        m['round'] = r
        m['elapsed_sec'] = round(time.time() - t0, 1)
        history.append(m)
        print(f"  R{r:02d} | mAP50={m['mAP_50']:.4f} | mAP50-95={m['mAP_50_95']:.4f} | "
              f"P={m['precision']:.3f} R={m['recall']:.3f} | {m['elapsed_sec']}s")

    final_path = scen_dir / f'final_{scenario_name}.pt'
    shutil.copy2(global_w_path, final_path)
    with open(scen_dir / 'history.json', 'w') as f:
        json.dump({
            'scenario': scenario_name,
            'privacy_params': privacy,
            'computed_epsilon': privacy['epsilon'],
            'delta': DELTA_DP,
            'rounds': ROUNDS,
            'local_epochs': LOCAL_EPOCHS,
            'client_sizes': CLIENT_SIZES,
            'history': history,
        }, f, indent=2, default=str)
    return history, final_path

all_histories = {}
for scen in SCENARIOS:
    h, _ = run_scenario(scen)
    all_histories[scen] = h

## 7b. Ablation Diagnostik Collapse (R2 — Pembimbing II)

**Tujuan (R2):** membuktikan apakah collapse DP adalah **bug/artefak pipeline**
(magnitudo noise, BatchNorm, sharp-minimum) atau **fenomena DP fundamental**.
Sampai ablation ini membuktikan sebaliknya, klaim tetap "pada konfigurasi ini"
(lihat R3).

**Unit ablation** = 1 update lokal (1 klien representatif) + DP pada delta + eval.
Ini unit terkecil tempat collapse muncul, jadi murah & decisive. Tiap konfigurasi
diulang `ABLATION_SEEDS` kali (≥3 seed, sesuai R2).

| Kode | Menguji | Ekspektasi bila collapse = artefak |
|------|---------|-------------------------------------|
| A0 | baseline (C=0, σ=0) | mAP tinggi (kontrol) |
| A1/A2 | **clipping-only** (σ=0, C=10 / C=1) | ~baseline → clipping bukan penyebab |
| A3–A6 | **noise + variasi C** (noise/elem = σ·C) | mAP pulih saat C↓ → magnitudo noise penyebabnya |
| B1/B2 | **LR lebih kecil** (1e-3 / 1e-4) | delta lebih kecil → cek apakah collapse berkurang |
| B3 | **frozen backbone** | lindungi fitur pretrained → cek pemulihan |
| C1/C1b | **BatchNorm→GroupNorm** (+DP / baseline GN) | bandingkan GN+DP vs GN-baseline |

**Verdict logic** (dicetak sel analisis): jika clip-only ≈ baseline DAN mAP pulih
saat C diturunkan → collapse = **artefak magnitudo noise pipeline**, BUKAN batas
mendasar DP. Jika collapse bertahan di semua mitigasi tanpa NaN → lebih dekat ke
fenomena nyata.

> **Waktu Colab:** tiap unit ≈ 1× train lokal + 1× eval. Grid penuh (12 konfig ×
> 3 seed = 36 run) bisa beberapa jam. Untuk smoke test: set `ABLATION_SEEDS=[0]`
> dan/atau kurangi `ABLATION_GRID` dulu.


In [ ]:
# === R2 ABLATION — helper functions ===
# Defensive imports: aman bila kernel restart antar sesi.
from ultralytics import YOLO
import torch, math, copy, json, time, shutil, random as _random
import numpy as np
from pathlib import Path
import torch.nn as nn

ABLATION_OUT  = Path('/content/sim_runs/ablation'); ABLATION_OUT.mkdir(parents=True, exist_ok=True)
ABL_CLIENT_YAML  = str(CLIENT_DATA_ROOT / 'client_1' / 'data.yaml')  # 1 klien representatif
ABL_LOCAL_EPOCHS = LOCAL_EPOCHS

def convert_bn_to_gn(module, num_groups=32):
    """Ganti seluruh BatchNorm2d -> GroupNorm secara rekursif (uji hipotesis
    BatchNorm sbg penyebab collapse). Jumlah grup disesuaikan agar habis membagi
    channel. Catatan: konversi pada model BN-pretrained menghapus running-stats,
    sehingga baseline GN (C1b) WAJIB dibandingkan terhadap GN+DP (C1), bukan
    terhadap baseline BN."""
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            c = child.num_features
            g = num_groups
            while c % g != 0 and g > 1:
                g -= 1
            setattr(module, name, nn.GroupNorm(g, c))
        else:
            convert_bn_to_gn(child, num_groups)
    return module

def _count_norm(m):
    n_bn = sum(isinstance(x, nn.BatchNorm2d) for x in m.modules())
    n_gn = sum(isinstance(x, nn.GroupNorm)  for x in m.modules())
    return n_bn, n_gn

def eval_live(model, val_yaml, img=640, batch=16):
    """Eval model in-memory (tanpa reload dari disk) supaya arsitektur GN tidak
    di-rebuild jadi BN oleh YOLO(yaml)."""
    r = model.val(data=str(val_yaml), imgsz=img, batch=batch, device=0,
                  workers=2, verbose=False, plots=False, save_json=False)
    return {'mAP_50': float(r.box.map50), 'mAP_50_95': float(r.box.map),
            'precision': float(r.box.mp), 'recall': float(r.box.mr)}

def run_ablation_unit(name, clip_norm=0.0, noise_mult=0.0, lr=0.01,
                      freeze=0, norm_swap=False, seed=42):
    """Satu unit ablation. Semua operasi in-memory agar konversi GN bertahan.
    Return dict metrik + diagnostik noise/sinyal + flag NaN."""
    torch.manual_seed(seed); np.random.seed(seed); _random.seed(seed)
    work = ABLATION_OUT / f'{name}_s{seed}'; work.mkdir(parents=True, exist_ok=True)

    model = YOLO(W0_LOCAL)
    if norm_swap:
        convert_bn_to_gn(model.model)
    nbn, ngn = _count_norm(model.model)
    w0_state = yolo_state(model)

    # --- 1 update lokal dgn lr/freeze/arsitektur tsb ---
    targs = dict(data=ABL_CLIENT_YAML, epochs=ABL_LOCAL_EPOCHS, imgsz=IMG_SIZE,
                 batch=BATCH_SIZE, optimizer='SGD', lr0=lr, device=0, workers=2,
                 cache=False, project=str(work), name='t', exist_ok=True,
                 verbose=False, plots=False, save=False, seed=seed)
    if freeze and freeze > 0:
        targs['freeze'] = freeze
    model.train(**targs)
    local_state = yolo_state(model)

    # --- delta + DP pada delta ---
    delta = {k: (local_state[k] - w0_state[k]) for k in w0_state}
    delta, dp = apply_dp_to_delta(delta, clip_norm, noise_mult)
    ratio = (dp['noise_per_elem'] / (dp['signal_per_elem'] + 1e-12)) if dp['noise_per_elem'] > 0 else 0.0

    # --- rekonstruksi w0+delta, cek NaN, eval in-memory ---
    final = {k: (w0_state[k].float() + delta[k].float()).to(w0_state[k].dtype) for k in w0_state}
    has_nan = any(torch.isnan(v.float()).any().item() for v in final.values())
    load_state(model, final)
    m = eval_live(model, GLOBAL_VAL_YAML, img=IMG_SIZE, batch=BATCH_SIZE)

    eps = epsilon_from_sigma(noise_mult, ROUNDS) if noise_mult > 0 else float('inf')
    rec = {'name': name, 'seed': seed, 'clip_norm': clip_norm, 'noise_mult': noise_mult,
           'lr': lr, 'freeze': freeze, 'norm_swap': norm_swap, 'n_bn': nbn, 'n_gn': ngn,
           'mAP_50': round(m['mAP_50'], 5), 'mAP_50_95': round(m['mAP_50_95'], 5),
           'precision': round(m['precision'], 5), 'recall': round(m['recall'], 5),
           'pre_clip_norm': round(dp['pre_norm'], 4), 'clipped': dp['clipped'],
           'signal_per_elem': dp['signal_per_elem'], 'noise_per_elem': dp['noise_per_elem'],
           'noise_signal_ratio': round(ratio, 3), 'has_nan': bool(has_nan),
           'epsilon': eps}
    del model; torch.cuda.empty_cache()
    shutil.rmtree(work, ignore_errors=True)
    return rec

print('R2 ablation helpers ready (run_ablation_unit, convert_bn_to_gn, eval_live).')


In [ ]:
# === R2 ABLATION — grid & driver ===
# Tiap entri = 1 konfigurasi. Subset/kurangi bila waktu Colab mepet.
ABLATION_GRID = [
    dict(name='A0_baseline',        clip_norm=0.0,  noise_mult=0.0),
    dict(name='A1_cliponly_C10',    clip_norm=10.0, noise_mult=0.0),
    dict(name='A2_cliponly_C1',     clip_norm=1.0,  noise_mult=0.0),
    dict(name='A3_noise_C10_s5e3',  clip_norm=10.0, noise_mult=0.005),
    dict(name='A4_noise_C1_s5e3',   clip_norm=1.0,  noise_mult=0.005),
    dict(name='A5_noise_C01_s5e3',  clip_norm=0.1,  noise_mult=0.005),
    dict(name='A6_noise_C10_s1e3',  clip_norm=10.0, noise_mult=0.001),
    dict(name='B1_lr1e3',           clip_norm=10.0, noise_mult=0.005, lr=0.001),
    dict(name='B2_lr1e4',           clip_norm=10.0, noise_mult=0.005, lr=0.0001),
    dict(name='B3_frozenbackbone',  clip_norm=10.0, noise_mult=0.005, freeze=11),
    dict(name='C1_groupnorm_dp',    clip_norm=10.0, noise_mult=0.005, norm_swap=True),
    dict(name='C1b_groupnorm_base', clip_norm=0.0,  noise_mult=0.0,   norm_swap=True),
]
ABLATION_SEEDS = [0, 1, 2]   # >=3 seed (R2). Smoke test: [0].

ablation_records = []
t_start = time.time()
for cfg in ABLATION_GRID:
    for sd in ABLATION_SEEDS:
        tag = f"{cfg['name']} | seed={sd}"
        try:
            rec = run_ablation_unit(seed=sd, **cfg)
            flag = ' [NaN!]' if rec['has_nan'] else ''
            print(f"  {tag:34s} -> mAP50={rec['mAP_50']:.4f} "
                  f"noise/sig={rec['noise_signal_ratio']:>6}x{flag}")
        except Exception as e:
            rec = {**cfg, 'seed': sd, 'error': str(e)[:200]}
            print(f"  {tag:34s} -> ERROR: {str(e)[:80]}")
        ablation_records.append(rec)

import pandas as pd
abl_df = pd.DataFrame(ablation_records)
abl_df.to_csv(ABLATION_OUT / 'ablation_results.csv', index=False)
print(f"\nSelesai {len(ablation_records)} run dalam {(time.time()-t_start)/60:.1f} menit.")
print(f"Tersimpan: {ABLATION_OUT / 'ablation_results.csv'}")


In [ ]:
# === R2 ABLATION — analisis & verdict ===
import pandas as pd, numpy as np

abl_df = pd.read_csv(ABLATION_OUT / 'ablation_results.csv')
ok = abl_df[~abl_df.get('mAP_50', pd.Series(dtype=float)).isna()] if 'mAP_50' in abl_df else abl_df

# Agregasi per konfigurasi (mean/std antar seed)
agg = (ok.groupby('name')
         .agg(mAP_mean=('mAP_50','mean'), mAP_std=('mAP_50','std'),
              nan_rate=('has_nan','mean'),
              noise_sig=('noise_signal_ratio','mean'))
         .reset_index())
agg['mAP_std'] = agg['mAP_std'].fillna(0.0)
print('=== Ringkasan ablation (rata-rata antar seed) ===')
print(agg.to_string(index=False,
      formatters={'mAP_mean':'{:.4f}'.format,'mAP_std':'{:.4f}'.format,
                  'nan_rate':'{:.2f}'.format,'noise_sig':'{:.1f}'.format}))

def gm(n):
    r = agg[agg['name']==n]
    return float(r['mAP_mean'].iloc[0]) if len(r) else float('nan')

base   = gm('A0_baseline')
clip10 = gm('A1_cliponly_C10')
nC10   = gm('A3_noise_C10_s5e3')
nC1    = gm('A4_noise_C1_s5e3')
nC01   = gm('A5_noise_C01_s5e3')
gn_dp  = gm('C1_groupnorm_dp')
gn_bs  = gm('C1b_groupnorm_base')
any_nan = bool(agg['nan_rate'].fillna(0).gt(0).any())

print('\n=== VERDICT (otomatis, untuk pembahasan sidang) ===')
print(f'- baseline mAP={base:.4f}')
# 1. clipping
if not np.isnan(clip10):
    verdict_clip = 'BUKAN penyebab (clip-only ~ baseline)' if clip10 > 0.5*base else 'BERKONTRIBUSI (clip-only sudah menurunkan mAP)'
    print(f'- clipping-only C=10: mAP={clip10:.4f} -> {verdict_clip}')
# 2. magnitudo noise vs C
if not any(np.isnan(x) for x in [nC10,nC1,nC01]):
    recover = (nC01 - nC10)
    if nC10 < 0.1*base and nC01 > 0.5*base:
        print(f'- variasi C (noise/elem=σ·C): mAP pulih {nC10:.3f}->{nC01:.3f} saat C 10->0.1')
        print('  => COLLAPSE DIDORONG MAGNITUDO NOISE PIPELINE (artefak), BUKAN batas mendasar DP.')
    elif nC01 < 0.1*base:
        print(f'- variasi C: collapse BERTAHAN walau C kecil ({nC01:.3f}) -> indikasi fenomena lebih dalam (cek B/C).')
    else:
        print(f'- variasi C: hasil campuran (C10={nC10:.3f}, C1={nC1:.3f}, C0.1={nC01:.3f}); periksa manual.')
# 3. BatchNorm vs GroupNorm
if not any(np.isnan(x) for x in [gn_dp,gn_bs]):
    print(f'- GroupNorm: baseline GN={gn_bs:.3f} vs GN+DP={gn_dp:.3f} '
          f'({"pulih" if gn_dp>0.5*gn_bs else "tetap collapse"} relatif thd baseline GN)')
# 4. NaN
print(f'- NaN terdeteksi di suatu konfigurasi: {any_nan} '
      f'({"ADA bug numerik pipeline" if any_nan else "tidak ada NaN -> collapse bukan sekadar overflow"})')

agg.to_csv(ABLATION_OUT / 'ablation_summary.csv', index=False)

# Salin ke Drive bila ter-mount
try:
    dst = Path('/content/drive/MyDrive/fedx-palm/federated_simulation')
    dst.mkdir(parents=True, exist_ok=True)
    for f in ['ablation_results.csv','ablation_summary.csv']:
        shutil.copy2(ABLATION_OUT / f, dst / f)
    print(f'\nDisalin ke Drive: {dst}')
except Exception as e:
    print(f'\n(skip copy Drive: {e})')

print('\nCatatan: tempel ringkasan + verdict ke Bab 4.3 (analisis penyebab) & R2.')


## 8. Summary & plot privacy-utility trade-off

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

rows = []
for scen, hist in all_histories.items():
    last = hist[-1]
    rows.append({
        'scenario':     scen,
        'sigma':        PRIVACY_SCENARIOS[scen]['noise_multiplier'],
        'epsilon':      round(PRIVACY_SCENARIOS[scen]['epsilon'], 1),
        'mAP@0.5':      round(last['mAP_50'], 4),
        'mAP@0.5:0.95': round(last['mAP_50_95'], 4),
        'Precision':    round(last['precision'], 4),
        'Recall':       round(last['recall'], 4),
    })
df = pd.DataFrame(rows).sort_values('sigma').reset_index(drop=True)
print(df.to_string(index=False))
df.to_csv(SIM_OUTPUT / 'summary_all_scenarios.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel kiri: konvergensi mAP@0.5 per ronde
for scen, hist in all_histories.items():
    rounds = [h['round'] for h in hist]
    maps   = [h['mAP_50'] for h in hist]
    sig    = PRIVACY_SCENARIOS[scen]['noise_multiplier']
    axes[0].plot(rounds, maps, marker='o', label=f'{scen} (σ={sig})')
axes[0].set_xlabel('Communication Round'); axes[0].set_ylabel('mAP@0.5')
axes[0].set_title('Konvergensi per skenario'); axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)

# Panel kanan: trade-off vs sigma (skala log). baseline (sigma=0) diplot terpisah.
labels = list(all_histories.keys())
pts = [(PRIVACY_SCENARIOS[s]['noise_multiplier'], all_histories[s][-1]['mAP_50'], s)
       for s in labels]
nz = sorted([(x, y, s) for x, y, s in pts if x > 0])
if nz:
    xs, ys, ss = zip(*nz)
    axes[1].plot(xs, ys, marker='s', color='#2563eb')
    for x, y, s in zip(xs, ys, ss):
        axes[1].annotate(f"{s}\n(ε={PRIVACY_SCENARIOS[s]['epsilon']:.0f})",
                         (x, y), textcoords='offset points', xytext=(4, 4), fontsize=7)
    axes[1].set_xscale('log')
# garis baseline (tanpa DP) sbg acuan
base = [y for x, y, s in pts if x == 0]
if base:
    axes[1].axhline(base[0], ls='--', color='gray', label=f'baseline (σ=0): {base[0]:.3f}')
    axes[1].legend(fontsize=8)
axes[1].set_xlabel('σ (noise multiplier, skala log)'); axes[1].set_ylabel('Final mAP@0.5')
axes[1].set_title('Privacy-Utility Trade-off'); axes[1].grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig(SIM_OUTPUT / 'privacy_utility_tradeoff.png', dpi=150)
plt.show()

## 9. XAI: Grad-CAM++ + Average Drop + FRR

Diaplikasikan pada model **final tiap skenario**. Karena Grad-CAM++ pada detection head Ultralytics tidak standar, di sini pakai **proxy berbasis confidence map**: kalau prediksi tetap kuat setelah masking area objek → model tidak fokus di objek yang benar.

In [14]:
import cv2, numpy as np

def xai_metrics(model_path, val_dir, n_samples=20):
    m = YOLO(model_path)
    imgs = [p for p in Path(val_dir).iterdir() if p.suffix.lower() in {'.jpg', '.png'}][:n_samples]
    drops, frrs = [], []
    for p in imgs:
        img = cv2.imread(str(p))
        if img is None: continue
        H, W = img.shape[:2]
        r = m.predict(source=str(p), imgsz=640, verbose=False, device=0)
        if not r or len(r[0].boxes) == 0: continue
        boxes = r[0].boxes
        conf_o = float(boxes.conf.max().item())
        # Saliency proxy: union of confident box regions
        sal = np.zeros((H, W), dtype=np.float32)
        for box, c in zip(boxes.xyxy.cpu().numpy(), boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = map(int, box)
            sal[max(0,y1):min(H,y2), max(0,x1):min(W,x2)] = np.maximum(
                sal[max(0,y1):min(H,y2), max(0,x1):min(W,x2)], float(c))
        blurred = cv2.GaussianBlur(sal, (51, 51), 0)
        inside = blurred[sal > 0].sum()
        total = blurred.sum() + 1e-12
        frrs.append(float(inside / total))
        mask = (sal > 0).astype(np.uint8)
        masked = img.copy()
        masked[mask > 0] = 0
        r2 = m.predict(source=masked, imgsz=640, verbose=False, device=0)
        conf_m = float(r2[0].boxes.conf.max().item()) if (r2 and len(r2[0].boxes)) else 0.0
        drops.append(max(0.0, conf_o - conf_m) / max(1e-6, conf_o) * 100)
    return {
        'average_drop_pct': float(np.mean(drops)) if drops else float('nan'),
        'frr':              float(np.mean(frrs)) if frrs else float('nan'),
        'n_samples':        len(drops),
    }

xai_results = {}
val_img_dir = str(Path(RAW_DIR) / 'valid' / 'images')
for scen in SCENARIOS:
    final_path = SIM_OUTPUT / scen / f'final_{scen}.pt'
    if not final_path.exists():
        print(f'[skip] {scen}: final model not found')
        continue
    print(f'XAI on {scen}...')
    xai_results[scen] = xai_metrics(str(final_path), val_img_dir, n_samples=20)
    print(f'  AD={xai_results[scen]["average_drop_pct"]:.2f}% | FRR={xai_results[scen]["frr"]:.3f}')

with open(SIM_OUTPUT / 'xai_summary.json', 'w') as f:
    json.dump(xai_results, f, indent=2)

XAI on baseline...
  AD=95.10% | FRR=0.962
XAI on weak...
  AD=nan% | FRR=nan
XAI on moderate...
  AD=nan% | FRR=nan
XAI on strong...
  AD=nan% | FRR=nan


In [15]:
# Merge XAI into summary
for i, row in df.iterrows():
    scen = row['scenario']
    if scen in xai_results:
        df.at[i, 'Average Drop (%)'] = round(xai_results[scen]['average_drop_pct'], 2)
        df.at[i, 'FRR'] = round(xai_results[scen]['frr'], 3)
print(df.to_string(index=False))
df.to_csv(SIM_OUTPUT / 'summary_all_scenarios.csv', index=False)

scenario  epsilon  sigma  mAP@0.5  mAP@0.5:0.95  Precision  Recall  Average Drop (%)   FRR
baseline      inf  0.000   0.9945        0.8973     0.9934  0.9945              95.1 0.962
    weak      8.0  0.005   0.0000        0.0000     0.0000  0.0000               NaN   NaN
moderate      4.0  0.010   0.0000        0.0000     0.0000  0.0000               NaN   NaN
  strong      1.0  0.020   0.0000        0.0000     0.0000  0.0000               NaN   NaN


## 10. Save semua hasil ke Drive

In [16]:
from pathlib import Path

# === Save final: Drive di Colab, lokal di Windows/Linux ===
if IS_COLAB:
    drive_dir = Path("/content/drive/MyDrive/fedx-palm/federated_simulation")
else:
    drive_dir = Path("/content/saved_runs")    # = D:\content\saved_runs di Windows
drive_dir.mkdir(parents=True, exist_ok=True)

# Copy summary & plot
for f in ["summary_all_scenarios.csv", "privacy_utility_tradeoff.png", "xai_summary.json",
          "privacy_budget_derivation.csv"]:
    src = SIM_OUTPUT / f
    if src.exists():
        shutil.copy2(src, drive_dir / f)
        print(f"OK  {drive_dir / f}")

# Copy per-scenario final models + history
for scen in SCENARIOS:
    sd = SIM_OUTPUT / scen
    if not sd.exists(): continue
    out = drive_dir / scen
    out.mkdir(exist_ok=True)
    for f in [f"final_{scen}.pt", "history.json"]:
        s = sd / f
        if s.exists():
            shutil.copy2(s, out / f)
            print(f"OK  {out / f}")

print(f"\nSemua tersimpan di: {drive_dir.absolute()}")


✓ /content/drive/MyDrive/fedx-palm/federated_simulation/summary_all_scenarios.csv
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/privacy_utility_tradeoff.png
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/xai_summary.json
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/baseline/final_baseline.pt
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/baseline/history.json
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/weak/final_weak.pt
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/weak/history.json
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/moderate/final_moderate.pt
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/moderate/history.json
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/strong/final_strong.pt
✓ /content/drive/MyDrive/fedx-palm/federated_simulation/strong/history.json

Semua tersimpan di: /content/drive/MyDrive/fedx-palm/federated_simulation


## 11. Validasi hipotesis tesis (Bab 4.7)

Setelah simulasi selesai, cek:

- **H1** — baseline (ε=∞) mAP@0.5 > 0.80? → lihat baris `baseline` di `summary_all_scenarios.csv`
- **H2** — mAP turun monoton dari baseline → weak → moderate → strong? → lihat plot kanan (privacy-utility trade-off)
- **H3** — Average Drop pada baseline > 20% dan menurun seiring ε mengecil? → lihat kolom `Average Drop (%)` di summary

## Catatan untuk replikasi tesis penuh

Notebook ini sengaja pakai konfigurasi demo (`ROUNDS=20`, `LOCAL_EPOCHS=3`) supaya bisa selesai dalam 1–2 session Colab Pro. Untuk replikasi penuh sesuai tesis Tabel 3.5:

```python
ROUNDS = 100
LOCAL_EPOCHS = 5
```

Total runtime full: ~5–7 hari di T4. Realistis kalau Anda punya akses GPU lebih lama atau jalankan di beberapa session paralel.